## Setup — Connect to the UGOT

Import the UGOT SDK and helper libraries, then:
- **Initialize** the UGOT at its IP address (opens a gRPC connection on port 50051).
- **Load vision models** that will be used later in the notebook.

Available models loaded here:
| Model | Purpose |
|---|---|
| `color_recognition` | Identifies dominant colors in the camera frame |
| `word_recognition` | Reads printed text (OCR) |
| `line_recognition` | Detects lines for line-following tasks |
| `face_recognition` | Identifies registered faces by name |

---
In case you get any import errors, please run the following while connected to wifi. You will only need to run it once, then feel free to delete. This will install all the necessary packages into your virtual environment (as long as you have selected the right kernel).

In [7]:
pip install torch torchvision ultralytics-thop opencv-python

Note: you may need to restart the kernel to use updated packages.


In [8]:
pip show torch

Name: torch
Version: 2.10.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /Users/bachnguyen/Desktop/National-Youth-Tech-Championship-2026/venv/lib/python3.13/site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: torchvision, ultralytics, ultralytics-thop
Note: you may need to restart the kernel to use updated packages.


In [1]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

In [6]:
import time
import importlib
import cv2
import numpy as np
from ugot import ugot
import pose_yolo

# Reload the pose control module so any edits to pose_yolo.py take effect
# without restarting the kernel.
importlib.reload(pose_yolo)
from pose_yolo import run_pose_control_inline

# Utilities for showing live camera frames directly inside the notebook
from IPython.display import display, clear_output, Image
from PIL import Image as Image2

# Create the robot controller object
got = ugot.UGOT()

# Connect to the UGOT robot using its IP address.
# Change this string if your robot is on a different IP.
got.initialize("192.168.88.1")

# Load the built-in computer vision models that will be used later in the notebook.
# You can load additional models here if needed — just add the model name to the list.
got.load_models(
    [
        "color_recognition",  # detects dominant colors
        "word_recognition",  # OCR: reads printed text
        "line_recognition",  # for line-following tasks
        "face_recognition",  # identifies registered faces by name
    ]
)

# Select the default line-tracking mode.
# 0 = single-line mode (follows one line at a time)
got.set_track_recognition_line(0)

# Open the camera stream so later functions can read live frames
got.open_camera()

print("Done!")

192.168.88.1:50051
Done!


## Live Camera Preview

Before running any vision-based behavior, it's useful to check that the camera feed is working and the robot can see what you expect.

The `display_frame()` helper captures a single frame from the robot's camera and displays it inline in the notebook. It handles the full decoding pipeline:
- `read_camera_data()` returns the raw frame as compressed bytes
- `np.frombuffer` + `cv2.imdecode` decode those bytes into a NumPy image array
- `cv2.cvtColor` converts from BGR (OpenCV's default color order) to RGB (what the notebook display expects)

The loop below calls `display_frame()` as fast as possible, with `clear_output(wait=True)` replacing each frame before drawing the next — giving you a live video preview directly inside the notebook.

Press the **Stop** button or interrupt the kernel to exit.

> **Heads up:** Decoding and rendering frames is computationally expensive. Running the camera preview alongside other code (e.g. inside a line-following or face-recognition loop) can slow down the robot's response time. Use this cell to verify your camera setup, then remove the `display_frame()` call before running any time-sensitive behaviors.

In [7]:
got.open_camera()


def display_frame():
    frame = got.read_camera_data()
    if frame is not None:  # Checks if the frame is real!
        nparr = np.frombuffer(frame, np.uint8)  # Reads in the image data
        data = cv2.imdecode(nparr, cv2.IMREAD_COLOR)  # Turns numbers into color
        frame_rgb = cv2.cvtColor(data, cv2.COLOR_BGR2RGB)

        display(Image2.fromarray(frame_rgb))
        clear_output(wait=True)

In [8]:
try:
    while True:
        display_frame()

except KeyboardInterrupt:
    print("Done!")

Done!


## Pose Control

Pose control lets you drive the robot using your body movements, no controller needed!

It works by running a **YOLO pose-estimation model** on a webcam feed. The model tracks your skeleton (shoulders, wrists, hips, etc.) and translates your arm positions into movement commands:

| Gesture | Command |
|---|---|
| Both hands raised above shoulders | **FORWARD** |
| Both hands lowered below shoulders | **BACKWARD** |
| Only left hand raised | **LEFT** |
| Only right hand raised | **RIGHT** |
| Hands close together at shoulder height | **EXIT** (stops the program) |
| No clear gesture | **NONE** (robot stops) |

To avoid jittery movement from small pose variations, there is a **deadzone**: a band around your shoulders where hand position is ignored. You'll see it as a yellow shaded region in the camera preview.

The code for this lives in `pose_yolo.py`. The function `run_pose_control_inline(...)` is the main entry point; it handles the camera loop, pose detection, debouncing, and robot commands all in one call.

> **Tip:** If your robot isn't moving even though the camera window shows detections, double-check that `enable_robot=True`.

### `run_pose_control_inline` — Parameter Reference

Here is a full explanation of every parameter the function accepts:

---

#### Connection
| Parameter | Type | Default | Description |
|---|---|---|---|
| `robot_ip` | `str` | `'192.168.88.1'` | IP address of the UGOT robot. Must match the address shown on the robot's screen. Only used if `enable_robot=True` and no existing `got` object is passed in. |

---

#### Movement Speeds
These control how fast the robot moves in response to each gesture. All are integers on the robot's internal speed scale.

| Parameter | Type | Default | Description |
|---|---|---|---|
| `forward_speed` | `int` | `30` | Speed when both hands are raised (FORWARD gesture). |
| `backward_speed` | `int` | `30` | Speed when both hands are lowered (BACKWARD gesture). |
| `turn_speed` | `int` | `45` | Speed when turning left or right. |

---

#### Camera & Model
| Parameter | Type | Default | Description |
|---|---|---|---|
| `camera_index` | `int` | `0` | Which webcam to use. `0` is your computer's built-in camera; `1` would be the first external camera, etc. |
| `model_path` | `str` | `'yolov8n-pose.pt'` | Path to the YOLO pose model file. `yolov8n-pose.pt` is the smallest/fastest model. Larger models (e.g. `yolov8s-pose.pt`) are more accurate but slower. The file will be auto-downloaded on first use if not present. |

---

#### Deadzone Tuning
The deadzone is a band around your shoulder height where hand position is ignored. This prevents small wobbles from triggering unwanted commands.

Both margins are expressed as a **fraction of your torso length** (the distance from shoulder to hip), so they automatically scale with how far you are from the camera.

| Parameter | Type | Default | Description |
|---|---|---|---|
| `up_margin_factor` | `float` | `0.1` | How far **above** shoulder level a hand must be to count as "raised". A value of `0.6` means 60% of torso length above the shoulder. Increase this if FORWARD/LEFT/RIGHT triggers too easily. |
| `down_margin_factor` | `float` | `0.1` | How far **below** shoulder level a hand must be to count as "lowered". Increase this if BACKWARD triggers too easily. |
| `min_conf` | `float` | `0.3` | Minimum YOLO keypoint confidence to use a joint. Joints with confidence below this are ignored. Raise it (e.g. `0.5`) to reduce false detections; lower it if joints are frequently dropped. |

---

#### Behaviour
| Parameter | Type | Default | Description |
|---|---|---|---|
| `enable_robot` | `bool` | `True` | Set to `False` to run pose detection **without** moving the robot. Useful for testing and calibrating the deadzone before going live. |
| `debounce_frames` | `int` | `5` | Number of consecutive frames a gesture must appear before it is acted on. Higher values = more stable but slightly slower response. Lower values = more responsive but jitterier. |
| `max_frames` | `int or None` | `None` | Stop after this many frames. Useful for automated testing. `None` means run indefinitely until EXIT gesture or `KeyboardInterrupt`. |

### Pose Control — Test Run (robot disabled)

Use this cell to verify that pose detection is working before involving the robot.

Because `enable_robot=False`, the camera and model run normally but **no movement commands are sent**. You'll see your skeleton overlaid on the camera feed, and the current detected gesture printed in the top-left corner.

This is the safest way to:
- Check that your webcam is being read correctly
- Calibrate `up_margin_factor` / `down_margin_factor` so gestures feel natural
- Confirm the model is detecting your keypoints reliably

When you're happy with the detections, move on to the **Real Run** cell below.

In [ ]:
try:
    run_pose_control_inline(
        robot_ip="192.168.88.1",  # Robot IP (unused here since enable_robot=False)
        forward_speed=30,  # Speed for FORWARD gesture
        backward_speed=30,  # Speed for BACKWARD gesture
        turn_speed=45,  # Speed for LEFT / RIGHT gesture
        camera_index=0,  # 0 = built-in webcam
        model_path="yolov8n-pose.pt",  # Smallest, fastest YOLO pose model
        up_margin_factor=0.6,  # Hands must be 60% of torso length ABOVE shoulder to count as 'up'
        down_margin_factor=0.6,  # Hands must be 60% of torso length BELOW shoulder to count as 'down'
        min_conf=0.3,  # Ignore keypoints with less than 30% confidence
        enable_robot=False,  # <-- Robot is DISABLED: safe for testing
        debounce_frames=2,  # Gesture must appear in 2 consecutive frames before triggering
        max_frames=None,  # Run indefinitely (stop with KeyboardInterrupt or EXIT gesture)
    )

except KeyboardInterrupt:
    print("Done")

### Pose Control — Real Run (robot enabled)

Same as the test run above, but with `enable_robot=True` so the robot actually moves.

Make sure you have enough space around the robot before running this cell. Use the **EXIT** gesture (hands close together at shoulder height) to stop the program cleanly, or press the **Stop** button in the Jupyter toolbar.

In [ ]:
try:
    run_pose_control_inline(
        robot_ip="192.168.88.1",
        forward_speed=30,
        backward_speed=30,
        turn_speed=45,
        camera_index=0,
        model_path="yolov8n-pose.pt",
        up_margin_factor=0.6,
        down_margin_factor=0.6,
        min_conf=0.3,
        enable_robot=True,  # <-- Robot is ENABLED: it will now move!
        debounce_frames=2,
        max_frames=None,
    )

except KeyboardInterrupt:
    print("Done")

192.168.88.1:50051


## Face Recognition

The UGOT has a built-in **face recognition** model that can learn and identify people by name.

The workflow is:
1. **Register** a face with `face_recognition_add_name(name=...)`; point the camera at the person and call this.
2. **Detect** faces in real time with `get_face_recognition_total_info()`, which returns a list of `[name, center_x, center_y, height, width, area]` for each detected face.
3. **Delete** a registered face with `face_recognition_delete_name(name=...)` if needed.

The cells below walk through each step.

In [11]:
# Register a new face under a given name.
# Point the camera at the person's face, then run this cell.
# The robot will capture and store the face so it can recognize them later.
got.face_recognition_add_name(name="Ryan")  # Replace "placeholder" with the person's real name!

Face [Ryan] already exists


In [ ]:
# Delete a previously registered face by name.
# Useful for clearing test entries or correcting mistakes.
got.face_recognition_delete_name(name="placeholder")

In [ ]:
# Gets a list of all currently registered names
got.face_recognition_get_all_names()

### Live Face & Text Recognition Preview

This cell runs a live camera loop that overlays both OCR text results and face recognition results directly on the frame (handy for checking that both models are working before using them).

- **Green text (top):** Whatever printed text the OCR model can read in the frame
- **Green text (below):** The face data: `[name, center_x, center_y, height]`

Press the **Stop** button or interrupt the kernel to exit.

In [1]:
# Ensure the correct models are loaded and the camera is open
# !!!This will NOT be needed on the actual competition. It is here for testing purposes only.
got.load_models(["word_recognition", "face_recognition"])
got.open_camera()

try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Overlay any detected text in the top-left corner
        name = got.get_words_result()
        if name:
            cv2.putText(img, f"text: {name}", (20, 40), 0, 0.8, (0, 255, 0), 2)

        # Overlay face recognition data if any faces are detected.
        # Each face entry is: [name, center_x, center_y, height]
        faces = got.get_face_recognition_total_info()
        if faces:
            face = faces[0]  # Only look at the first detected face
            cv2.putText(img, f"face: {face}", (20, 70), 0, 0.8, (0, 255, 0), 2)

        # Encode the annotated frame as JPEG and display it inline in the notebook
        _, jpeg = cv2.imencode(".jpg", img)
        clear_output(wait=True)
        display(Image(data=jpeg.tobytes()))

except KeyboardInterrupt:
    print("Done")
    got.mecanum_stop()

NameError: name 'got' is not defined

### `face_find_and_approach` — Find a Person and Walk Up to Them

This function combines face recognition and movement to make the robot:
1. **Spin** in place until it spots the target person (by face or by reading their name tag)
2. **Approach** the person, keeping them centered in frame, until it gets close enough

**How the approach works:**
- The camera frame is 640 px wide, so the center is at x = 320.
- If the face center (`c_x`) is too far **left** of center, the robot strafes left while moving forward.
- If `c_x` is too far **right**, it strafes right while moving forward.
- If the face is roughly centered but the detected face **height** (`h`) is still small (meaning the robot is far away), it moves straight forward.
- Once the face height exceeds the `height` threshold (meaning the robot is close enough), it stops.
- If the face is lost mid-approach, the robot inches forward slowly until it reappears.

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `gap` | `10` | Pixel tolerance around the center (x = 320 ± gap). Within this band the robot won't strafe. |
| `target_name` | `"Stranger"` | The name to search for; must match either the face recognition label or text the OCR can read. |
| `turn_spd` | `15` | Speed used when spinning to search for the target. |
| `strafe_spd` | `10` | Sideways speed used to re-center the target in frame during approach. |
| `fwd_spd` | `10` | Forward speed used during approach. |
| `height` | `80` | How tall (in pixels) the face bounding box must be before the robot considers itself close enough to stop. Decrease this to stop further away; increase it to get closer. |
| `adjust_turn` | `10` | How far the robot turns to center itself after first spotting the target, in degree-units. Increase if the robot tends to approach at an angle. |

In [ ]:
def face_find_and_approach(
    gap=10, target_name="Stranger", turn_spd=15, strafe_spd=10, fwd_spd=10, height=80, adjust_turn=10
):
    """
    Spin until the target person is found, then approach them.

    Phase 1 — Search:
        The robot turns continuously, checking each frame for the target's
        face (via face_recognition) or name tag (via OCR). When found, it
        stops and does a small corrective turn to face them, then moves to Phase 2.

    Phase 2 — Approach:
        The robot drives toward the face, strafing left/right to keep it
        centered in frame, until the face appears large enough (close enough).
    """
    face_name = None  # Will hold the name from face recognition once a face is found

    try:
        # Phase 1: Spin and search
        while True:
            # Turn slowly to scan the environment
            got.mecanum_turn_speed(turn=3, speed=turn_spd)

            # Read text visible in the frame (e.g. a name tag)
            name = got.get_words_result()

            # Check for any recognized faces in the frame
            faces = got.get_face_recognition_total_info()
            if faces:
                face_name = faces[0][0]  # faces[0] = first face; [0] = its name

            # If either the OCR text or the face name matches our target, we found them!
            if name == target_name or face_name == target_name:
                got.mecanum_stop()
                print(f"Saw {target_name}!")

                # Small corrective turn to center the robot on the target
                # turn=3 is clockwise; times=10, unit=2 means ~10 degree-units
                got.mecanum_turn_speed_times(turn=3, speed=20, times=adjust_turn, unit=2)
                break  # Exit Phase 1, move on to Phase 2

        # Phase 2: Approach the target
        while True:
            name = got.get_words_result()
            faces = got.get_face_recognition_total_info()

            if not faces:
                # Lost the face; inch forward slowly to try to find it again
                got.mecanum_translate_speed(angle=0, speed=fwd_spd)
            else:
                c_x = faces[0][1]  # Horizontal center of the face in the frame (0–640 px)
                h = faces[0][3]  # Height of the face bounding box (proxy for distance)
                if h < height:
                    if c_x < 320 - gap:
                        # Face is too far LEFT — strafe left while moving forward
                        got.mecanum_move_xyz(
                            x_speed=-strafe_spd, y_speed=fwd_spd, z_speed=0
                        )
                    elif c_x > 320 + gap:
                        # Face is too far RIGHT — strafe right while moving forward
                        got.mecanum_move_xyz(x_speed=strafe_spd, y_speed=fwd_spd, z_speed=0)
                    else:
                        # Face is centered but still small (far away) — move straight forward
                        got.mecanum_move_xyz(x_speed=0, y_speed=fwd_spd, z_speed=0)
                else:
                    # Face is centered AND large enough — we've arrived!
                    got.mecanum_stop()
                    print(f"Reached {name}!")
                    break  # Done

            clear_output(wait=True)

        got.mecanum_stop()

    except KeyboardInterrupt:
        print("Done")
        got.mecanum_stop()

In [ ]:
# Run the find-and-approach behavior.
# The robot will spin until it sees "Coley" (by face or name tag),
# then drive up to them and stop when it gets close.
face_find_and_approach(
    gap=15, target_name="Coley", turn_spd=10, strafe_spd=10, fwd_spd=10, height=120, adjust_turn=10
)

## Real-Time Text (Word) Recognition

The UGOT's **OCR model** can read printed text visible in the camera frame in real time.

`get_words_result()` returns a string of any detected text, or an empty string if nothing is readable.

The first cell below is a simple **monitor**: it just prints whatever text the robot can see, refreshing each frame. This is useful for checking legibility before using OCR in a behavior.

The second cell shows a **reactive behavior**: the robot listens for the words `"LEFT"` or `"RIGHT"` and turns accordingly — a simple way to give the robot text-based commands.

In [ ]:
try:
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # Refresh output each frame so only the latest result is shown.
        # wait=True prevents flickering by waiting for new output before clearing.
        clear_output(wait=True)

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

### Text-Commanded Turning

Hold up a sign saying **LEFT** or **RIGHT** in front of the camera and the robot will turn to match.

- `mecanum_turn_speed_times(turn=2, ...)` turns **counter-clockwise** (left)
- `mecanum_turn_speed_times(turn=3, ...)` turns **clockwise** (right)
- `speed=30` is the turn speed; `times=45, unit=2` controls how far it turns

In [ ]:
try:
    while True:
        # Read any text currently visible in the camera frame.
        text = got.get_words_result()

        print("Text:", text)

        # React to specific command words
        if text == "LEFT":
            # Turn counter-clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=2, speed=30, times=45, unit=2)
            break
        elif text == "RIGHT":
            # Turn clockwise by ~45 degrees
            got.mecanum_turn_speed_times(turn=3, speed=30, times=45, unit=2)
            break

        # Refresh output each frame so only the latest result is shown.
        clear_output(wait=True)

    print("Done")

except KeyboardInterrupt:
    # Safety stop if the cell is interrupted manually.
    got.mecanum_stop()
    print("Done")

## Line Following + Text-Commanded Turns

This section combines two skills from earlier (**line following** and **OCR text recognition**) into a single autonomous command:

1. **Follow the line** until it reaches an intersection or the line ends
2. **Stop and wait** for a text command (`"LEFT"` or `"RIGHT"`)
3. **Turn** in the commanded direction
4. **Follow the line again** until it ends

This is useful for navigation tasks (for example, a robot that follows a line and takes a text-signposted turn at a junction)

The `line_follow()` helper function is defined first, then the main behavior loop follows.

### `line_follow()` — Helper Function

This small helper wraps the robot's line-tracking API into a single convenient call. Each time it's called it:
- Reads the current line detection from the robot
- Calculates a rotation speed proportional to how far off-center the line is
- Sends a combined forward + rotation movement command
- Returns the detection info so the caller can decide what to do next

**Parameters:**
| Parameter | Default | Description |
|---|---|---|
| `mult` | `0.25` | Multiplier that converts the line offset into a rotation speed. Higher = sharper corrections; lower = gentler steering. |
| `speed` | `35` | Forward speed while following the line. |

**Return values:** `(line_type, x, y)`
| Value | Description |
|---|---|
| `line_type` | `0` = no line detected, `1` = normal line, `2` = y-intersection (fork in the path) |

In [ ]:
def line_follow(mult=0.25, speed=35):
    """Follow the detected line by turning proportionally to the line offset."""
    # Read line-tracking information from the robot.
    # `offset` tells how far the detected line is from the center.
    # `type` describes the line/intersection pattern detected.
    # `x` and `y` are the detected line position coordinates.
    offset, line_type, _, _ = got.get_single_track_total_info()

    # Convert the line offset into a turning speed.
    # Larger offset -> stronger rotation to re-center on the line.
    rotation_speed = int(offset * mult)

    # Move forward while rotating to stay aligned with the line.
    got.mecanum_move_xyz(x_speed=0, y_speed=speed, z_speed=rotation_speed)

    # Return the detection info so the main program can make decisions.
    return line_type

### Main Behavior — Follow, Turn, Follow

The behavior runs in three sequential phases:

**Phase 1 — Follow to the junction:**
The robot follows the line slowly until `line_type` is `0` (line lost) or `2` (y-intersection detected), then stops. The intersection is the decision point.

**Phase 2 — Wait for a text command:**
The robot holds still and polls the OCR model every frame. It waits until it reads `"LEFT"` or `"RIGHT"`, executes the corresponding ~45 degree turn, and moves on.

**Phase 3 — Follow to the end:**
The robot follows the line again (at a slightly higher speed than Phase 1) until the line disappears (`line_type == 0`), then stops cleanly.

In [ ]:
# Set line tracking to single-line mode before starting
got.set_track_recognition_line(0)

try:
    # Phase 1: Follow the line to the junction
    while True:
        got.set_track_recognition_line(
            line_type=0
        )  # Ensure single-line mode each frame
        line_type, x, y = line_follow(mult=0.25, speed=15)  # Follow slowly

        if line_type in [0, 2]:  # 0 = no line detected, 2 = y-intersection (junction reached)
            got.mecanum_stop()  # Stop at the junction before deciding which way to go
            break

    # Phase 2: Wait for a text-commanded turn direction
    while True:
        text = got.get_words_result()  # Read any text currently visible in the frame

        if text == "LEFT":
            # Turn counter-clockwise ~45 degrees, then continue to Phase 3
            got.mecanum_turn_speed_times(turn=2, speed=30, times=45, unit=2)
            break
        elif text == "RIGHT":
            # Turn clockwise ~45 degrees, then continue to Phase 3
            got.mecanum_turn_speed_times(turn=3, speed=30, times=45, unit=2)
            break
        # If neither word is seen yet, loop and keep checking

    # Phase 3: Follow the line to the end
    while True:
        got.set_track_recognition_line(
            line_type=0
        )  # Ensure single-line mode each frame
        line_type, x, y = line_follow(mult=0.25, speed=35)  # Follow at full speed

        if line_type == 0:  # Line has disappeared — end of the track
            got.mecanum_stop()
            break

    print("Done")

except KeyboardInterrupt:
    got.mecanum_stop()
    print("Done")